# Knowledge Graph Extraction - Pre-Selected Texts

## Overview
This notebook extracts knowledge graphs from 4 pre-selected Northern Ireland education texts:
1. Madden (2011) CCEA Chp 3 - Changing Relationships (Option 2)
2. Madden (2011) CCEA Chp 2 - Peace, War, Neutrality (Option 1)
3. Updated Johnston&Johnston (Option 1)
4. Doherty (2001) - Northern Ireland since c.1960 (Option 2)

## Approach
- Uses **refined ontology** with separate definitions for Option 1 and Option 2
- **Enhanced extractor** with system/user message separation
- Validates entities and relationships against ontology
- Extracts discourse patterns and perspectives

In [12]:
# Cell 1: Imports and Setup
import os
import sys
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Add scripts folder to path
sys.path.insert(0, '../scripts')

# Import our custom modules
from refined_ontology import RefinedOntology, CorpusOption, get_ontology
from enhanced_knowledge_extractor import NIKnowledgeExtractor, KnowledgeTriple
from kg_utilities import (
    SimpleRateLimiter,
    TextChunker,
    format_extraction_summary,
    load_target_documents,
    prepare_corpus_data
)

# LLM imports
from openai import OpenAI

# Standard libraries
import json
import pickle
from datetime import datetime
from typing import List, Dict

print("✅ All imports successful!")
print(f"📅 Notebook run: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ All imports successful!
📅 Notebook run: 2026-02-10 02:03:34


In [13]:
# Cell 2: Configuration

# Load environment variables
load_dotenv(override=True)

# Configuration
class Config:
    # API Keys
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
    
    # Model settings
    LLM_MODEL = 'gpt-4.1-mini-2025-04-14'
    
    # Chunking settings
    CHUNK_SIZE = 800  # Smaller for short texts
    CHUNK_OVERLAP = 150
    
    # Rate limiting
    API_DELAY = 1.0  # seconds between requests
    
    # Paths
    DATA_PATH = '../outputs/cleaned_text_data.csv'
    OUTPUT_PATH = '../graph_data'
    
    # Target files
    TARGET_FILES = [
        "Madden (2011) CCEA revision guide Chp 3. Changing Relationships.docx",
        "Madden (2011) ccea revision guide chp 2 peace war neutrality.docx",
        "Updated Johnston&Johnston no textboxes.docx",
        "Doherty (2001) Northern Ireland since c.1960 .docx"
    ]

config = Config()

# Initialize OpenAI client
if config.OPENAI_API_KEY:
    openai_client = OpenAI(api_key=config.OPENAI_API_KEY)
    print("✅ OpenAI client initialized")
    print(f"🎯 Using model: {config.LLM_MODEL}")
else:
    print("❌ ERROR: OPENAI_API_KEY not found in environment")
    print("💡 Set it in your .env file")

# Create output directory
os.makedirs(config.OUTPUT_PATH, exist_ok=True)
print(f"📁 Output directory: {config.OUTPUT_PATH}")

print("\n✅ Configuration complete!")

✅ OpenAI client initialized
🎯 Using model: gpt-4.1-mini-2025-04-14
📁 Output directory: ../graph_data

✅ Configuration complete!


In [14]:
# Cell 3: Load Data and Target Documents

print("📂 Loading dataset...")
df = pd.read_csv(config.DATA_PATH)
print(f"✅ Dataset loaded: {len(df):,} total rows")
print(f"   Columns: {df.columns.tolist()}")

# Load the 4 target documents (all chunks)
print("\n" + "="*70)
print("📚 Loading Target Documents")
print("="*70)

documents = load_target_documents(df, config.TARGET_FILES)

print("\n" + "="*70)
print("📊 Summary")
print("="*70)
print(f"Documents loaded: {len(documents)}/4")

# Show breakdown by religious group
option1_docs = [d for d in documents.values() if d['religious_group'] == 'option1']
option2_docs = [d for d in documents.values() if d['religious_group'] == 'option2']

print(f"\nOption 1 (Protestant/Unionist): {len(option1_docs)} documents")
for doc in option1_docs:
    print(f"  - {doc['filename'][:60]}...")
    print(f"    {doc['total_chunks']} chunks | {doc['total_words']:,} words")

print(f"\nOption 2 (Catholic/Nationalist): {len(option2_docs)} documents")
for doc in option2_docs:
    print(f"  - {doc['filename'][:60]}...")
    print(f"    {doc['total_chunks']} chunks | {doc['total_words']:,} words")

total_words = sum(d['total_words'] for d in documents.values())
print(f"\nTotal words across all documents: {total_words:,}")

📂 Loading dataset...
✅ Dataset loaded: 4,704 total rows
   Columns: ['filename', 'file_type', 'religious_group', 'content', 'content_length', 'word_count', 'chunk_id', 'chunk_count', 'has_url_content', 'has_ai_summary', 'source_url', 'publication_policy_year', 'cleaned_content']

📚 Loading Target Documents
✅ Loaded: Madden (2011) CCEA revision guide Chp 3. Changing Relationships.docx
   Group: option2 | 77 chunks | 9,580 total words
✅ Loaded: Madden (2011) ccea revision guide chp 2 peace war neutrality.docx
   Group: option1 | 75 chunks | 9,608 total words
✅ Loaded: Updated Johnston&Johnston no textboxes.docx
   Group: option1 | 134 chunks | 22,452 total words
✅ Loaded: Doherty (2001) Northern Ireland since c.1960 .docx
   Group: option2 | 3 chunks | 371 total words

📊 Summary
Documents loaded: 4/4

Option 1 (Protestant/Unionist): 2 documents
  - Madden (2011) ccea revision guide chp 2 peace war neutrality...
    75 chunks | 9,608 words
  - Updated Johnston&Johnston no textboxes.docx..

In [15]:
# Cell 4: Prepare Corpus Data (Already Chunked!)

print("📋 Organizing pre-chunked data by perspective...")
corpus_data = prepare_corpus_data(documents)

print("\n" + "="*70)
print("📊 Corpus Data Summary")
print("="*70)
print(f"Option 1 chunks: {len(corpus_data['option1']['chunks'])}")
print(f"Option 2 chunks: {len(corpus_data['option2']['chunks'])}")
print(f"Total chunks: {len(corpus_data['option1']['chunks']) + len(corpus_data['option2']['chunks'])}")

# Show sample chunks
print("\n📝 Sample chunks:")
if corpus_data['option1']['chunks']:
    sample = corpus_data['option1']['chunks'][0]
    print(f"\nOption 1 chunk example (first 200 chars):")
    print(f"  {sample[:200]}...")

if corpus_data['option2']['chunks']:
    sample = corpus_data['option2']['chunks'][0]
    print(f"\nOption 2 chunk example (first 200 chars):")
    print(f"  {sample[:200]}...")

📋 Organizing pre-chunked data by perspective...

📄 Madden (2011) CCEA revision guide Chp 3. Changing Relationsh...
   Group: option2
   Chunks: 77

📄 Madden (2011) ccea revision guide chp 2 peace war neutrality...
   Group: option1
   Chunks: 75

📄 Updated Johnston&Johnston no textboxes.docx...
   Group: option1
   Chunks: 134

📄 Doherty (2001) Northern Ireland since c.1960 .docx...
   Group: option2
   Chunks: 3

📊 Corpus Data Summary
Option 1 chunks: 209
Option 2 chunks: 80
Total chunks: 289

📝 Sample chunks:

Option 1 chunk example (first 200 chars):
  war relationship outbreak second world war september established government act act partitioned part governed separate southern later became irish free state act gave parliament control area health tr...

Option 2 chunk example (first 200 chars):
  relation republic emergence development established government act act partitioned county southern later became irish free state population reacted new arrangement unionist mostly protestan

In [16]:
# Cell 5: Initialize Knowledge Extractor

print("🤖 Initializing Knowledge Extractor...")

# Initialize extractor
extractor = NIKnowledgeExtractor(
    client=openai_client,
    model=config.LLM_MODEL
)

# Initialize rate limiter
rate_limiter = SimpleRateLimiter(delay_seconds=config.API_DELAY)

print("\n✅ Extractor ready!")
print(f"   Model: {config.LLM_MODEL}")
print(f"   Rate limit: {config.API_DELAY}s between requests")
print("   Ontology: Auto-detected per corpus")

# Show ontology info
print("\n📚 Ontology Overview:")
ontology = RefinedOntology()

print("\nOption 1:")
opt1_labels = ontology.get_all_labels(CorpusOption.OPTION_1)
opt1_rels = ontology.get_relationships(CorpusOption.OPTION_1)
print(f"  Entity types: {len(opt1_labels)}")
print(f"  Relationship types: {len(opt1_rels)}")

print("\nOption 2:")
opt2_labels = ontology.get_all_labels(CorpusOption.OPTION_2)
opt2_rels = ontology.get_relationships(CorpusOption.OPTION_2)
print(f"  Entity types: {len(opt2_labels)}")
print(f"  Relationship types: {len(opt2_rels)}")

🤖 Initializing Knowledge Extractor...
Knowledge extractor initialized with model: gpt-4.1-mini-2025-04-14

✅ Extractor ready!
   Model: gpt-4.1-mini-2025-04-14
   Rate limit: 1.0s between requests
   Ontology: Auto-detected per corpus

📚 Ontology Overview:

Option 1:
  Entity types: 43
  Relationship types: 123

Option 2:
  Entity types: 46
  Relationship types: 131


In [17]:
# Cell 6: Extract Knowledge - Option 1 (Protestant/Unionist)

print("\n" + "="*70)
print("🔍 EXTRACTING KNOWLEDGE GRAPH - OPTION 1")
print("="*70)

start_time = datetime.now()

option1_triples = extractor.extract_from_corpus(
    chunks=corpus_data['option1']['chunks'],
    metadata=corpus_data['option1']['metadata'],
    corpus_name='option1',
    save_progress=True,
    rate_limiter=rate_limiter
)

elapsed_time = (datetime.now() - start_time).total_seconds()

print("\n✅ Option 1 extraction complete!")
print(f"   Time elapsed: {elapsed_time:.1f}s")
print(f"   Triples extracted: {len(option1_triples)}")

# Show sample triples
if option1_triples:
    print("\n📝 Sample triples:")
    for i, triple in enumerate(option1_triples[:3]):
        print(f"\n  {i+1}. {triple.subject} --[{triple.predicate}]--> {triple.object}")
        print(f"     Types: {triple.subject_type} --> {triple.object_type}")
        print(f"     Confidence: {triple.confidence:.2f}")


🔍 EXTRACTING KNOWLEDGE GRAPH - OPTION 1

Starting extraction from option1...
Total chunks: 209

📚 Ontology set for: OPTION_1
   Entity types: 43
   Relationship types: 123


Extracting option1:   0%|          | 1/209 [00:02<09:49,  2.83s/it]

  ⚠️  Invalid subject_type: Events
  ⚠️  Invalid subject_type: Entitys


Extracting option1:   1%|          | 2/209 [00:12<22:50,  6.62s/it]

  ⚠️  Invalid subject_type: Entitys
  ⚠️  Invalid subject_type: Entitys
  ⚠️  Invalid subject_type: Entitys
  ⚠️  Invalid subject_type: Entitys
  ⚠️  Invalid subject_type: Entitys
  ⚠️  Invalid subject_type: Events
  ⚠️  Invalid subject_type: Figures
  ⚠️  Invalid subject_type: Figures
  ⚠️  Invalid subject_type: Entitys
  ⚠️  Invalid subject_type: Entitys
  ⚠️  Invalid subject_type: Entitys
  ⚠️  Invalid subject_type: Entitys


...

  ⚠️  Invalid predicate: Know
  ⚠️  Invalid predicate: Ask
  ⚠️  Invalid predicate: Ask
  ⚠️  Invalid predicate: Vote


Extracting option1: 100%|█████████▉| 208/209 [19:09<00:08,  8.83s/it]

  ⚠️  Invalid subject_type: Entitys
  ⚠️  Invalid subject_type: Entitys
  ⚠️  Invalid subject_type: Entitys
  ⚠️  Invalid subject_type: Entitys
  ⚠️  Invalid subject_type: Entitys
  ⚠️  Invalid subject_type: Entitys
  ⚠️  Invalid subject_type: Entitys
  ⚠️  Invalid subject_type: Entitys
  ⚠️  Invalid subject_type: Entitys


Extracting option1: 100%|██████████| 209/209 [19:14<00:00,  5.52s/it]


✅ Extraction complete for option1!
  Extracted: 669 knowledge relations
  Success rate: 100.0%
  Total cost: $0.089

🔧 Normalizing entity names...
✅ Entity names normalized
  💾 Progress saved: ../graph_data/kg_triples_option1_final_20260210_022313.pkl

📋 Invalid predicates report saved: ../graph_data/invalid_predicates_option1_20260210_022313.json
   Total invalid: 219
   Unique predicates: 154

   Top 10 invalid predicates:
      Pay                  :   4 times
      Get                  :   4 times
      Denounce             :   3 times
      Meet                 :   3 times
      Accept               :   3 times
      Go To                :   3 times
      Target               :   3 times
      Drop                 :   3 times
      Come                 :   3 times
      Form                 :   3 times

✅ Option 1 extraction complete!
   Time elapsed: 1154.3s
   Triples extracted: 669

📝 Sample triples:

  1. Sir James Craig --[Become]--> prime
     Types: Figure --> Entity
     

In [18]:
# Cell 7: Extract Knowledge - Option 2 (Catholic/Nationalist)

print("\n" + "="*70)
print("🔍 EXTRACTING KNOWLEDGE GRAPH - OPTION 2")
print("="*70)

start_time = datetime.now()

option2_triples = extractor.extract_from_corpus(
    chunks=corpus_data['option2']['chunks'],
    metadata=corpus_data['option2']['metadata'],
    corpus_name='option2',
    save_progress=True,
    rate_limiter=rate_limiter
)

elapsed_time = (datetime.now() - start_time).total_seconds()

print("\n✅ Option 2 extraction complete!")
print(f"   Time elapsed: {elapsed_time:.1f}s")
print(f"   Triples extracted: {len(option2_triples)}")

# Show sample triples
if option2_triples:
    print("\n📝 Sample triples:")
    for i, triple in enumerate(option2_triples[:3]):
        print(f"\n  {i+1}. {triple.subject} --[{triple.predicate}]--> {triple.object}")
        print(f"     Types: {triple.subject_type} --> {triple.object_type}")
        print(f"     Confidence: {triple.confidence:.2f}")


🔍 EXTRACTING KNOWLEDGE GRAPH - OPTION 2

Starting extraction from option2...
Total chunks: 80

📚 Ontology set for: OPTION_2
   Entity types: 46
   Relationship types: 131


Extracting option2:   0%|          | 0/80 [00:00<?, ?it/s]

  📊 210 requests | 1166s elapsed | 0.2 req/s


Extracting option2:   1%|▏         | 1/80 [00:06<08:58,  6.81s/it]

  ⚠️  Invalid predicate: Delight


...

  ⚠️  Invalid predicate: Headed
  ⚠️  Invalid predicate: Headed
  ⚠️  Invalid predicate: Deal
  ⚠️  Invalid predicate: Deal
  ⚠️  Invalid predicate: Deal
  ⚠️  Invalid predicate: Occur
  ⚠️  Invalid predicate: Clearly Recognise
  ⚠️  Invalid predicate: Majority
  📊 280 requests | 1566s elapsed | 0.2 req/s


Extracting option2:  94%|█████████▍| 75/80 [07:04<00:26,  5.38s/it]

  ⚠️  Invalid predicate: Engage

  📊 Progress: 75/80 chunks
     Extracted: 285 relations
     Cost so far: $0.122
  💾 Progress saved: ../graph_data/kg_triples_option2_progress_75_20260210_023017.pkl
  📊 285 requests | 1590s elapsed | 0.2 req/s


Extracting option2: 100%|██████████| 80/80 [07:20<00:00,  5.50s/it]


✅ Extraction complete for option2!
  Extracted: 293 knowledge relations
  Success rate: 100.0%
  Total cost: $0.123

🔧 Normalizing entity names...
✅ Entity names normalized
  💾 Progress saved: ../graph_data/kg_triples_option2_final_20260210_023033.pkl

📋 Invalid predicates report saved: ../graph_data/invalid_predicates_option2_20260210_023033.json
   Total invalid: 306
   Unique predicates: 213

   Top 10 invalid predicates:
      Pay                  :   4 times
      Reveal               :   4 times
      Try                  :   4 times
      Get                  :   4 times
      Denounce             :   3 times
      Meet                 :   3 times
      Accept               :   3 times
      Go To                :   3 times
      Target               :   3 times
      Drop                 :   3 times

✅ Option 2 extraction complete!
   Time elapsed: 440.2s
   Triples extracted: 293

📝 Sample triples:

  1. Irish Free State --[Become]--> Éire
     Types: Entity --> Entity
     C

In [19]:
# Cell 8: Final Summary and Statistics

print("\n" + "="*70)
print("📊 FINAL EXTRACTION SUMMARY")
print("="*70)

# Combined statistics
total_triples = len(option1_triples) + len(option2_triples)

print(f"\nDocuments processed: {len(documents)}")
print(f"Total chunks: {len(corpus_data['option1']['chunks']) + len(corpus_data['option2']['chunks'])}")
print(f"\nOption 1 triples: {len(option1_triples)}")
print(f"Option 2 triples: {len(option2_triples)}")
print(f"Total triples: {total_triples}")

# Get rate limiter stats
rate_stats = rate_limiter.get_summary()
print(f"\nAPI calls: {rate_stats['total_requests']}")
print(f"Total cost: ${rate_stats['total_cost']:.3f}")
print(f"Time elapsed: {rate_stats['elapsed_seconds']}s")

# Show extractor stats
print("\n" + "="*70)
extractor.print_final_summary()

# Save combined results
print("\n💾 Saving results...")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

all_results = {
    'option1': option1_triples,
    'option2': option2_triples,
    'metadata': {
        'extraction_date': timestamp,
        'model': config.LLM_MODEL,
        'target_files': config.TARGET_FILES,
        'total_triples': total_triples,
        'statistics': extractor.extraction_stats
    }
}

output_file = os.path.join(config.OUTPUT_PATH, f'preselected_kg_{timestamp}.pkl')
with open(output_file, 'wb') as f:
    pickle.dump(all_results, f)

print(f"✅ Results saved: {output_file}")
print("\n🎉 Extraction complete!")


📊 FINAL EXTRACTION SUMMARY

Documents processed: 4
Total chunks: 289

Option 1 triples: 669
Option 2 triples: 293
Total triples: 962

API calls: 289
Total cost: $0.123
Time elapsed: 1605s


📊 FINAL KNOWLEDGE GRAPH EXTRACTION SUMMARY
Total chunks processed:  289
Successful extractions:  289
Failed extractions:      0
Success rate:            100.0%
Total relations:         962
Avg relations/chunk:     3.3
API calls made:          289
Total cost:              $0.123

⚠️  VALIDATION STATISTICS
Invalid predicates found: 306
Unique invalid predicates: 213

Top 5 most common:
  Pay                  :   4 times
  Reveal               :   4 times
  Try                  :   4 times
  Get                  :   4 times
  Denounce             :   3 times

💾 Saving results...
✅ Results saved: ../graph_data/preselected_kg_20260210_023033.pkl

🎉 Extraction complete!


In [20]:
# Cell 9: Quick Analysis - Compare Perspectives

from collections import Counter

print("\n" + "="*70)
print("🔍 COMPARATIVE ANALYSIS")
print("="*70)

# Analyze relationships by perspective
print("\n📊 Top Relationship Types:")

option1_rels = [t.predicate for t in option1_triples]
option2_rels = [t.predicate for t in option2_triples]

print("\nOption 1 (Protestant/Unionist):")
for rel, count in Counter(option1_rels).most_common(10):
    print(f"  {rel:20s} : {count:3d}")

print("\nOption 2 (Catholic/Nationalist):")
for rel, count in Counter(option2_rels).most_common(10):
    print(f"  {rel:20s} : {count:3d}")

# Analyze entities
print("\n📊 Top Entities Mentioned:")

option1_entities = [t.subject for t in option1_triples] + [t.object for t in option1_triples]
option2_entities = [t.subject for t in option2_triples] + [t.object for t in option2_triples]

print("\nOption 1:")
for entity, count in Counter(option1_entities).most_common(10):
    print(f"  {entity:30s} : {count:3d}")

print("\nOption 2:")
for entity, count in Counter(option2_entities).most_common(10):
    print(f"  {entity:30s} : {count:3d}")

# Find unique vs shared entities
opt1_set = set(option1_entities)
opt2_set = set(option2_entities)
shared = opt1_set & opt2_set
opt1_only = opt1_set - opt2_set
opt2_only = opt2_set - opt1_set

print("\n📊 Entity Overlap:")
print(f"  Shared entities: {len(shared)}")
print(f"  Option 1 only: {len(opt1_only)}")
print(f"  Option 2 only: {len(opt2_only)}")

if shared:
    print(f"\n  Shared entities: {list(shared)[:10]}")


🔍 COMPARATIVE ANALYSIS

📊 Top Relationship Types:

Option 1 (Protestant/Unionist):
  Want                 :  29
  Build                :  25
  Remain               :  25
  Become               :  24
  Attack               :  22
  Declare              :  21
  Cause                :  20
  Fear                 :  20
  Provide              :  19
  Support              :  17

Option 2 (Catholic/Nationalist):
  React                :  17
  Result               :  14
  Resign               :  13
  Organise             :  12
  Oppose               :  10
  Support              :  10
  Fear                 :   8
  Increase             :   8
  Demand               :   8
  Allow                :   7

📊 Top Entities Mentioned:

Option 1:
  Éire                           :  63
  Éamon de Valera                :  33
  Éamon De Valera                :  32
  Winston Churchill              :  32
  Britain                        :  29
  Belfast Blitz                  :  29
  Neutrality                  

In [21]:
# Cell 10: Export for Neo4j or Further Analysis

print("\n" + "="*70)
print("💾 EXPORTING DATA")
print("="*70)

# Export to JSON for easy viewing
json_file = os.path.join(config.OUTPUT_PATH, f'preselected_kg_{timestamp}.json')

json_data = {
    'option1': [
        {
            'subject': t.subject,
            'predicate': t.predicate,
            'object': t.object,
            'subject_type': t.subject_type,
            'object_type': t.object_type,
            'confidence': t.confidence,
            'source_corpus': t.source_corpus
        }
        for t in option1_triples
    ],
    'option2': [
        {
            'subject': t.subject,
            'predicate': t.predicate,
            'object': t.object,
            'subject_type': t.subject_type,
            'object_type': t.object_type,
            'confidence': t.confidence,
            'source_corpus': t.source_corpus
        }
        for t in option2_triples
    ],
    'metadata': all_results['metadata']
}

with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(json_data, f, indent=2, ensure_ascii=False)

print(f"✅ JSON exported: {json_file}")

# Export to CSV for easy analysis in Excel/R
csv_file = os.path.join(config.OUTPUT_PATH, f'preselected_kg_{timestamp}.csv')

all_triples_list = []
for t in option1_triples + option2_triples:
    all_triples_list.append({
        'corpus': t.source_corpus,
        'subject': t.subject,
        'subject_type': t.subject_type,
        'predicate': t.predicate,
        'object': t.object,
        'object_type': t.object_type,
        'confidence': t.confidence,
        'chunk_id': t.chunk_id
    })

df_export = pd.DataFrame(all_triples_list)
df_export.to_csv(csv_file, index=False)

print(f"✅ CSV exported: {csv_file}")
print(f"\n📊 Exported {len(all_triples_list)} total triples")
print("\n🎉 All exports complete!")


💾 EXPORTING DATA
✅ JSON exported: ../graph_data/preselected_kg_20260210_023033.json
✅ CSV exported: ../graph_data/preselected_kg_20260210_023033.csv

📊 Exported 962 total triples

🎉 All exports complete!


In [22]:
# Cell 11: Load Knowledge Graph into Neo4j (v2 - weighted edges)

# Import from the new module (make sure neo4j_loader_v2.py is in ../scripts/)
sys.path.insert(0, '../scripts')  # already done in Cell 1, but just in case
from neo4j_loader_v2 import (
    Neo4jConfig, connect_to_neo4j, clear_database, create_indexes,
    load_entities, load_relationships_weighted, verify_data, print_sample_queries
)

print("\n" + "="*70)
print("🔗 CONNECTING TO NEO4J")
print("="*70)

neo4j_config = Neo4jConfig()
driver = connect_to_neo4j(neo4j_config)

if driver:
    try:
        # Clear existing data
        clear_database(driver, neo4j_config.DATABASE)
        
        # Create indexes
        create_indexes(driver, neo4j_config.DATABASE)
        
        # Load entities
        entities = load_entities(driver, neo4j_config.DATABASE, option1_triples, option2_triples)
        
        # Load relationships WITH EDGE MERGING (v2 change)
        load_relationships_weighted(driver, neo4j_config.DATABASE, option1_triples, option2_triples)
        
        # Verify
        verify_data(driver, neo4j_config.DATABASE)
        
        # Print sample queries
        print_sample_queries()
        
        print("\n" + "="*70)
        print("🎉 KNOWLEDGE GRAPH LOADED INTO NEO4J!")
        print("="*70)
        
    except Exception as e:
        print(f"\n❌ Error loading data: {e}")
        import traceback
        traceback.print_exc()
    
    finally:
        driver.close()
        print("\n✅ Neo4j connection closed")
else:
    print("\n❌ Cannot load data without Neo4j connection")
    print("   Make sure Neo4j is running and credentials are correct")


🔗 CONNECTING TO NEO4J
✅ Connected to Neo4j at neo4j+s://a052fe1a.databases.neo4j.io

⚠️  Clearing existing data...
✅ Database cleared

📊 Creating indexes...
✅ Index on entity names created

📦 Loading entities...
   Found 881 unique entities
✅ Created 881 entity nodes

🔗 Loading relationships (with edge merging)...
   Raw edges: 962
   After merging: 923 (removed 39 duplicates)
   Edges with weight > 1: 26
   Max edge weight: 7
✅ Created 923 merged relationships

🔍 Verifying data...
✅ Nodes in Neo4j: 881
✅ Merged relationships: 923
   (representing 962 original edges)

   By corpus:
      Protestant/Unionist       : 580 entities
      Catholic/Nationalist      : 258 entities
      Shared                    :  43 entities

   Edge weight stats:
      Edges with weight > 1: 26
      Max weight: 7
      Avg weight (of multi): 2.5

🔍 USEFUL NEO4J QUERIES (Weighted Graph)

View all nodes:

MATCH (n:Entity)
RETURN n
LIMIT 25
        

View relationships with weight:

MATCH (a:Entity)-[r:RELA

In [ ]:
# Cell 11: Load Knowledge Graph into Neo4j

from neo4j import GraphDatabase
import os

print("\n" + "="*70)
print("🔗 CONNECTING TO NEO4J")
print("="*70)

# Neo4j Configuration
class Neo4jConfig:
    # Update these with your Neo4j credentials
    URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
    USER = os.getenv('NEO4J_USER', 'neo4j')
    PASSWORD = os.getenv('NEO4J_PASSWORD', 'password')  # Change this!
    DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

neo4j_config = Neo4jConfig()

# Color scheme for visualization
CORPUS_COLORS = {
    'option1': '#E74C3C',  # Red (Protestant/Unionist)
    'option2': '#3498DB',  # Blue (Catholic/Nationalist)
    'both': '#2ECC71'       # Green (Shared)
}

CORPUS_LABELS = {
    'option1': 'Protestant/Unionist',
    'option2': 'Catholic/Nationalist',
    'both': 'Shared'
}

def connect_to_neo4j(config):
    """Connect to Neo4j database"""
    try:
        driver = GraphDatabase.driver(
            config.URI,
            auth=(config.USER, config.PASSWORD)
        )
        # Test connection
        driver.verify_connectivity()
        print(f"✅ Connected to Neo4j at {config.URI}")
        return driver
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        print("\n💡 Make sure Neo4j is running and credentials are correct")
        print("   Update .env file with:")
        print("   NEO4J_URI=bolt://localhost:7687")
        print("   NEO4J_USER=neo4j")
        print("   NEO4J_PASSWORD=your_password")
        return None

def clear_database(driver, database):
    """Clear existing data (optional - use with caution!)"""
    print("\n⚠️  Clearing existing data...")
    with driver.session(database=database) as session:
        # Delete all nodes and relationships
        session.run("MATCH (n) DETACH DELETE n")
        print("✅ Database cleared")

def create_indexes(driver, database):
    """Create indexes for better performance"""
    print("\n📊 Creating indexes...")
    with driver.session(database=database) as session:
        # Index on entity names
        try:
            session.run("CREATE INDEX entity_name IF NOT EXISTS FOR (n:Entity) ON (n.name)")
            print("✅ Index on entity names created")
        except Exception as e:
            print(f"⚠️  Index creation: {e}")

def load_entities(driver, database, option1_triples, option2_triples):
    """Load all entities as nodes"""
    print("\n📦 Loading entities...")
    
    # Collect all unique entities with their types and corpus
    entities = {}
    
    for triple in option1_triples:
        # Add subject
        if triple.subject not in entities:
            entities[triple.subject] = {
                'type': triple.subject_type,
                'corpora': set()
            }
        entities[triple.subject]['corpora'].add('option1')
        
        # Add object
        if triple.object not in entities:
            entities[triple.object] = {
                'type': triple.object_type,
                'corpora': set()
            }
        entities[triple.object]['corpora'].add('option1')
    
    for triple in option2_triples:
        # Add subject
        if triple.subject not in entities:
            entities[triple.subject] = {
                'type': triple.subject_type,
                'corpora': set()
            }
        entities[triple.subject]['corpora'].add('option2')
        
        # Add object
        if triple.object not in entities:
            entities[triple.object] = {
                'type': triple.object_type,
                'corpora': set()
            }
        entities[triple.object]['corpora'].add('option2')
    
    print(f"   Found {len(entities)} unique entities")
    
    # Load entities into Neo4j
    with driver.session(database=database) as session:
        created = 0
        for entity_name, entity_data in entities.items():
            # Determine corpus label and color
            corpora_list = list(entity_data['corpora'])
            if len(corpora_list) == 2:
                corpus = 'both'
            else:
                corpus = corpora_list[0]
            
            color = CORPUS_COLORS[corpus]
            corpus_label = CORPUS_LABELS[corpus]
            
            # Create entity node
            session.run("""
                MERGE (e:Entity {name: $name})
                SET e.type = $type,
                    e.corpus = $corpus,
                    e.corpus_label = $corpus_label,
                    e.color = $color,
                    e.occurrences = $occurrences
            """,
            name=entity_name,
            type=entity_data['type'],
            corpus=corpus,
            corpus_label=corpus_label,
            color=color,
            occurrences=len(corpora_list)
            )
            created += 1
        
        print(f"✅ Created {created} entity nodes")
    
    return entities

def load_relationships(driver, database, option1_triples, option2_triples):
    """Load all relationships"""
    print("\n🔗 Loading relationships...")
    
    all_triples = option1_triples + option2_triples
    
    with driver.session(database=database) as session:
        created = 0
        skipped = 0
        
        for triple in all_triples:
            try:
                corpus_color = CORPUS_COLORS[triple.source_corpus]
                corpus_label = CORPUS_LABELS[triple.source_corpus]
                
                # Create relationship
                session.run("""
                    MATCH (a:Entity {name: $subject})
                    MATCH (b:Entity {name: $object})
                    CREATE (a)-[r:RELATED {
                        type: $predicate,
                        confidence: $confidence,
                        corpus: $corpus,
                        corpus_label: $corpus_label,
                        color: $color,
                        chunk_id: $chunk_id
                    }]->(b)
                """,
                subject=triple.subject,
                object=triple.object,
                predicate=triple.predicate,
                confidence=triple.confidence,
                corpus=triple.source_corpus,
                corpus_label=corpus_label,
                color=corpus_color,
                chunk_id=triple.chunk_id
                )
                created += 1
                
            except Exception as e:
                skipped += 1
                if skipped <= 3:
                    print(f"  ⚠️  Skipped relationship: {e}")
        
        print(f"✅ Created {created} relationships")
        if skipped > 0:
            print(f"⚠️  Skipped {skipped} relationships")

def verify_data(driver, database):
    """Verify loaded data"""
    print("\n🔍 Verifying data...")
    
    with driver.session(database=database) as session:
        # Count nodes
        result = session.run("MATCH (n:Entity) RETURN count(n) as count")
        node_count = result.single()['count']
        
        # Count relationships
        result = session.run("MATCH ()-[r:RELATED]->() RETURN count(r) as count")
        rel_count = result.single()['count']
        
        # Count by corpus
        result = session.run("""
            MATCH (n:Entity)
            RETURN n.corpus as corpus, count(n) as count
            ORDER BY count DESC
        """)
        corpus_counts = {record['corpus']: record['count'] for record in result}
        
        print(f"✅ Nodes in Neo4j: {node_count:,}")
        print(f"✅ Relationships in Neo4j: {rel_count:,}")
        print(f"\n   By corpus:")
        for corpus, count in corpus_counts.items():
            label = CORPUS_LABELS.get(corpus, corpus)
            print(f"      {label:25s} : {count:3d} entities")

def print_sample_queries():
    """Print useful Cypher queries"""
    print("\n" + "="*70)
    print("📝 USEFUL NEO4J QUERIES")
    print("="*70)
    
    queries = {
        "View all nodes": """
MATCH (n:Entity)
RETURN n
LIMIT 25
        """,
        
        "View relationships": """
MATCH (a:Entity)-[r:RELATED]->(b:Entity)
RETURN a.name, r.type, b.name, r.corpus_label
LIMIT 25
        """,
        
        "Option 1 only": """
MATCH (a:Entity)-[r:RELATED]->(b:Entity)
WHERE r.corpus = 'option1'
RETURN a, r, b
        """,
        
        "Option 2 only": """
MATCH (a:Entity)-[r:RELATED]->(b:Entity)
WHERE r.corpus = 'option2'
RETURN a, r, b
        """,
        
        "Shared entities": """
MATCH (n:Entity)
WHERE n.corpus = 'both'
RETURN n.name, n.type
ORDER BY n.occurrences DESC
        """,
        
        "Most connected entities": """
MATCH (n:Entity)
RETURN n.name, n.type, n.corpus_label, 
       size((n)-[:RELATED]->()) as out_degree,
       size((n)<-[:RELATED]-()) as in_degree,
       size((n)-[:RELATED]-()) as total_degree
ORDER BY total_degree DESC
LIMIT 10
        """,
        
        "Find specific entity": """
MATCH (n:Entity)
WHERE n.name CONTAINS 'Paisley'
RETURN n
        """,
        
        "Relationships between two entities": """
MATCH path = (a:Entity)-[r:RELATED*1..2]-(b:Entity)
WHERE a.name CONTAINS 'IRA' AND b.name CONTAINS 'Unionist'
RETURN path
LIMIT 10
        """
    }
    
    for title, query in queries.items():
        print(f"\n{title}:")
        print(query)

# ============================================================================
# MAIN EXECUTION
# ============================================================================

# Connect to Neo4j
driver = connect_to_neo4j(neo4j_config)

if driver:
    try:
        # Optional: Clear existing data (comment out if you want to keep old data)
        clear_database(driver, neo4j_config.DATABASE)
        
        # Create indexes
        create_indexes(driver, neo4j_config.DATABASE)
        
        # Load entities
        entities = load_entities(driver, neo4j_config.DATABASE, option1_triples, option2_triples)
        
        # Load relationships
        load_relationships(driver, neo4j_config.DATABASE, option1_triples, option2_triples)
        
        # Verify
        verify_data(driver, neo4j_config.DATABASE)
        
        # Print sample queries
        print_sample_queries()
        
        print("\n" + "="*70)
        print("🎉 KNOWLEDGE GRAPH LOADED INTO NEO4J!")
        print("="*70)
        print(f"\n💡 Open Neo4j Browser at: {neo4j_config.URI.replace('bolt://', 'http://')}")
        print("   Default: http://localhost:7474")
        print("\n💡 Try these queries in Neo4j Browser:")
        print("   1. MATCH (n) RETURN n LIMIT 50")
        print("   2. MATCH p=()-[r:RELATED]->() RETURN p LIMIT 25")
        
    except Exception as e:
        print(f"\n❌ Error loading data: {e}")
        import traceback
        traceback.print_exc()
    
    finally:
        driver.close()
        print("\n✅ Neo4j connection closed")
else:
    print("\n❌ Cannot load data without Neo4j connection")
    print("   Make sure Neo4j is running and credentials are correct")